<a href="https://colab.research.google.com/github/d005810/ECAA08-Manufatura-Flexivel/blob/main/etapa-01-logica/10%20-%20Avaliacao%20Integrada%20do%20Modulo%201.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 10 - Notebook: Avaliação Integrada do Módulo 1 — SCADA-Core Segurança & Diagnóstico
## Célula de Manufatura Flexível (FMS)

Este notebook integra todos os módulos lógicos desenvolvidos:
1. **Discretização Proposicional** de sensores e atuadores da planta.
2. **Lógica de Intertravamento e Trip** do motor principal M-101.
3. **Motor Especialista de Inferência** (Forward Chaining) para diagnóstico de causa-raiz em tempo real.

In [1]:
from dataclasses import dataclass
from typing import Dict, List, Set, Tuple, Any

def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

class MapeadorProposicional:
    """Discretiza a telemetria do SCADA em proposições booleanas atômicas."""
    def extrair_proposicoes(self, telemetria: Dict[str, Any]) -> Dict[str, bool]:
        return {
            's_base': bool(telemetria.get('ZS-201', 0)),
            's_topo': bool(telemetria.get('ZS-202', 0)),
            'not_s_base': not bool(telemetria.get('ZS-201', 0)),
            'cor_r': telemetria.get('AS-201', 0.0) >= 5.0,
            'cor_g': telemetria.get('AS-202', 0.0) >= 5.0,
            'cor_b': telemetria.get('AS-203', 0.0) >= 5.0,
            'silo_vazio': bool(telemetria.get('LS-101', 0)),
            'peca_saida': bool(telemetria.get('ZS-101', 0)),
            'emergencia': bool(telemetria.get('HS-301', 0)),
            'caixa_1_cheia': telemetria.get('US-301', 0) >= 10,
            'caixa_2_cheia': telemetria.get('US-302', 0) >= 10,
            'caixa_3_cheia': telemetria.get('US-303', 0) >= 10,
            'peca_em_transito': bool(telemetria.get('ZS-101', 0)) or bool(telemetria.get('ZS-201', 0)),
            'modo_auto': bool(telemetria.get('AUTO', 1))
        }

@dataclass
class RegraProducao:
    id_regra: str
    antecedentes: Set[str]
    consequente: str
    descricao_diagnostico: str
    prioridade: int = 1

class BaseConhecimento:
    def __init__(self):
        self.regras: List[RegraProducao] = []

    def adicionar_regra(self, id_r: str, antecedentes: List[str], consequente: str, desc: str, prioridade: int = 1):
        self.regras.append(RegraProducao(id_r, set(antecedentes), consequente, desc, prioridade))

class MotorInferencia:
    def __init__(self, base_conhecimento: BaseConhecimento):
        self.bc = base_conhecimento

    def forward_chaining(self, fatos_iniciais: Set[str]) -> Tuple[Set[str], List[Dict[str, Any]]]:
        fatos_conhecidos = set(fatos_iniciais)
        historico = []
        passo = 1
        novos = True

        while novos:
            novos = False
            for regra in sorted(self.bc.regras, key=lambda r: r.prioridade, reverse=True):
                if regra.antecedentes.issubset(fatos_conhecidos) and regra.consequente not in fatos_conhecidos:
                    fatos_conhecidos.add(regra.consequente)
                    historico.append({
                        "Passo": passo,
                        "Regra": regra.id_regra,
                        "Diagnóstico": regra.descricao_diagnostico
                    })
                    passo += 1
                    novos = True
                    break
        return fatos_conhecidos, historico

class SCADACoreModulo1:
    """Orquestrador Integrado do Módulo 1 para a Manufatura Flexível."""
    def __init__(self):
        self.mapeador = MapeadorProposicional()
        self.bc = BaseConhecimento()

        self.bc.adicionar_regra("R-01", ["s_topo", "not_s_base"], "erro_geometria", "Inconsistência ZS-202 sem ZS-201", 8)
        self.bc.adicionar_regra("R-02", ["cor_r", "cor_g"], "ambiguidade_cor", "Conflito Óptico nos Canais de Cor AS-201/202", 7)
        self.bc.adicionar_regra("R-03", ["silo_vazio", "peca_saida"], "conflito_silo", "LS-101 Vazio com Peça Presente em ZS-101", 7)
        self.bc.adicionar_regra("R-04", ["caixa_1_cheia", "caixa_2_cheia", "caixa_3_cheia"], "transbordo_geral", "Todas as Caixas Cheias (10/10)", 9)
        self.bc.adicionar_regra("R-05", ["erro_geometria", "peca_em_transito"], "alarme_a1", "Disparo de Alarme Geral HS-302", 9)
        self.bc.adicionar_regra("R-06", ["alarme_a1", "modo_auto"], "trip_m101", "Desarme de Segurança da Esteira M-101", 10)

        self.motor = MotorInferencia(self.bc)

    def processar_ciclo_scan(self, telemetria: Dict[str, Any]) -> Dict[str, Any]:
        props = self.mapeador.extrair_proposicoes(telemetria)
        fatos_ativos = {k for k, v in props.items() if v}

        fatos_inf, trilha = self.motor.forward_chaining(fatos_ativos)

        trip_ativo = (
            props['emergencia'] or
            ('trip_m101' in fatos_inf) or
            ('transbordo_geral' in fatos_inf)
        )

        return {
            "Motor_M101_Ativo": not trip_ativo,
            "Trip_Ativo": trip_ativo,
            "Alarmes_Inferidos": [f for f in fatos_inf if f in ['erro_geometria', 'ambiguidade_cor', 'conflito_silo', 'transbordo_geral', 'alarme_a1', 'trip_m101']],
            "Trilha_Disparos": trilha
        }

core1 = SCADACoreModulo1()

telemetria_teste = {
    'ZS-201': 0,
    'ZS-202': 1,
    'AS-201': 8.5,
    'AS-202': 0.2,
    'AS-203': 0.1,
    'LS-101': 0,
    'ZS-101': 1,
    'HS-301': 0,
    'US-301': 3,
    'US-302': 4,
    'US-303': 2,
    'AUTO': 1
}

res = core1.processar_ciclo_scan(telemetria_teste)

print("=== RESULTADO DA AVALIAÇÃO INTEGRADA DO MÓDULO 1 ===")
print(f"Estado do Motor M-101 : {'LIGADO' if res['Motor_M101_Ativo'] else 'DESARMADO (TRIP)'}")
print(f"Status do Trip        : {res['Trip_Ativo']}")
print(f"Diagnósticos Finais   : {res['Alarmes_Inferidos']}")

print("\n--- Trilha de Execução (Forward Chaining) ---")
print(formatar_tabela(res['Trilha_Disparos']))

assert res["Trip_Ativo"] is True
assert "erro_geometria" in res["Alarmes_Inferidos"]
assert "trip_m101" in res["Alarmes_Inferidos"]

print("\n[OK] Avaliação Integrada do Módulo 1 concluída com 100% de sucesso!")

=== RESULTADO DA AVALIAÇÃO INTEGRADA DO MÓDULO 1 ===
Estado do Motor M-101 : DESARMADO (TRIP)
Status do Trip        : True
Diagnósticos Finais   : ['erro_geometria', 'trip_m101', 'alarme_a1']

--- Trilha de Execução (Forward Chaining) ---
Passo | Regra | Diagnóstico                          
------+-------+--------------------------------------
1     | R-01  | Inconsistência ZS-202 sem ZS-201     
2     | R-05  | Disparo de Alarme Geral HS-302       
3     | R-06  | Desarme de Segurança da Esteira M-101

[OK] Avaliação Integrada do Módulo 1 concluída com 100% de sucesso!
